In [31]:
import struct
from midiutil import MIDIFile

# Configuration
INPUT_BIN = "bpe_output_sample.bin"
OUTPUT_MIDI = "generated_music_smooth.mid"

CHUNK_SIZE = 36
DURATION_BYTES = 4

# MIDI Settings
TRACK = 0
CHANNEL = 0
TEMPO = 120
VOLUME = 100

def get_active_notes_from_mask(mask_bytes):
    """
    Decodes the 32-byte mask into a set of sounding MIDI notes (0-127).
    We assume the 32 bytes (256 bits) cover:
    - Bits 0-127: Keys currently pressed
    - Bits 128-255: Keys sustained (pedal)
    
    If EITHER is true, the note is sounding.
    """
    sounding_notes = set()
    
    # Check all 256 bits
    for byte_index, byte_val in enumerate(mask_bytes):
        for bit_index in range(8):
            if (byte_val >> bit_index) & 1:
                # Calculate absolute bit position (0-255)
                abs_bit = (byte_index * 8) + bit_index
                
                # Map back to MIDI Note (0-127)
                # If bit is > 127 (e.g. 188), it implies Note (188 - 128) = 60 is sustained
                midi_note = abs_bit % 128
                
                if 0 <= midi_note <= 127:
                    sounding_notes.add(midi_note)
                    
    return sounding_notes

def main():
    print(f"Reading from {INPUT_BIN}...")
    
    midi = MIDIFile(1)
    midi.addTempo(TRACK, 0, TEMPO)
    
    # STATE TRACKING
    # Dictionary: { midi_note: start_time_in_beats }
    active_notes_start_times = {}
    
    current_time_cursor = 0.0
    
    try:
        with open(INPUT_BIN, "rb") as f:
            while True:
                chunk = f.read(CHUNK_SIZE)
                if not chunk or len(chunk) < CHUNK_SIZE:
                    break

                # 1. Parse Duration
                duration = struct.unpack('<f', chunk[:DURATION_BYTES])[0]
                
                # 2. Parse Current State
                mask_payload = chunk[DURATION_BYTES:]
                current_sounding_set = get_active_notes_from_mask(mask_payload)
                
                # 3. Handle Note OFFs
                # Look for notes that were active, but are NOT in the new set
                ended_notes = []
                for note in list(active_notes_start_times.keys()):
                    if note not in current_sounding_set:
                        # The note has ended. Write it to MIDI.
                        start_time = active_notes_start_times[note]
                        note_duration = current_time_cursor - start_time
                        
                        # Optimization: Ignore extremely short blips (optional)
                        if note_duration > 0.01: 
                            midi.addNote(TRACK, CHANNEL, note, start_time, note_duration, VOLUME)
                        
                        del active_notes_start_times[note]

                # 4. Handle Note ONs
                # Look for notes in the new set that were NOT active before
                for note in current_sounding_set:
                    if note not in active_notes_start_times:
                        # New note started. Record the timestamp.
                        active_notes_start_times[note] = current_time_cursor

                # 5. Advance Time
                current_time_cursor += duration
                
        # 6. Cleanup: Close any notes still ringing at the end of file
        for note, start_time in active_notes_start_times.items():
            note_duration = current_time_cursor - start_time
            midi.addNote(TRACK, CHANNEL, note, start_time, note_duration, VOLUME)

    except FileNotFoundError:
        print(f"Error: Could not find file {INPUT_BIN}")
        return

    print(f"Writing to {OUTPUT_MIDI}...")
    with open(OUTPUT_MIDI, "wb") as output_file:
        midi.writeFile(output_file)
    print("Done!")

In [32]:
main()

Reading from bpe_output_sample.bin...
Writing to generated_music_smooth.mid...
Done!
